In [2]:
!pip install ultralytics filterpy lap


In [21]:
# First, install the missing package
!pip install scikit-image

# Then import your modules
import cv2
import numpy as np
from ultralytics import YOLO
from sort import Sort

In [23]:
model = YOLO("yolov8n.pt")   # lightweight & stable
tracker = Sort(max_age=30, min_hits=3, iou_threshold=0.3)


In [24]:
QUEUE_ROI = (200, 300, 1100, 650)  # x1, y1, x2, y2


In [22]:
def inside_roi(cx, cy, roi):
    x1, y1, x2, y2 = roi
    return x1 < cx < x2 and y1 < cy < y2


In [26]:
import cv2
import numpy as np

cap = cv2.VideoCapture("sam1.mp4")

# ---- Get video properties ----
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# ---- Video Writer ----
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # codec
out = cv2.VideoWriter("output_queue.mp4", fourcc, fps, (width, height))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    detections = []

    results = model(frame, stream=True, conf=0.4)

    for r in results:
        for box in r.boxes:
            cls = int(box.cls[0])
            if cls in [2, 3, 5, 7]:  # car, bike, bus, truck
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])
                detections.append([x1, y1, x2, y2, conf])

    if len(detections) > 0:
        tracked = tracker.update(np.array(detections))
    else:
        tracked = []

    queue_ids = set()

    # ---- Draw ROI ----
    x1, y1, x2, y2 = QUEUE_ROI
    cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 255, 0), 2)
    cv2.putText(frame, "QUEUE ROI", (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

    for obj in tracked:
        x1, y1, x2, y2, obj_id = obj.astype(int)
        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2

        if inside_roi(cx, cy, QUEUE_ROI):
            queue_ids.add(obj_id)
            color = (0, 0, 255)
        else:
            color = (0, 255, 0)

        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.circle(frame, (cx, cy), 4, (255, 255, 255), -1)
        cv2.putText(frame, f"ID {obj_id}",
                    (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    queue_length = len(queue_ids)

    cv2.putText(frame, f"Queue Length: {queue_length}",
                (40, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 1,
                (0, 0, 255), 3)

    # ---- WRITE FRAME TO VIDEO ----
    out.write(frame)

    # ---- SHOW VIDEO ----
    cv2.imshow("Queue Length Detection", frame)

    if cv2.waitKey(1) & 0xFF == 27:  # ESC
        break

cap.release()
out.release()
cv2.destroyAllWindows()



0: 384x640 3 persons, 9 cars, 1 bus, 196.2ms
Speed: 47.0ms preprocess, 196.2ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 9 cars, 1 bus, 175.5ms
Speed: 5.4ms preprocess, 175.5ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 8 cars, 1 bus, 205.1ms
Speed: 4.5ms preprocess, 205.1ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 12 cars, 2 buss, 181.2ms
Speed: 6.9ms preprocess, 181.2ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 8 cars, 1 bus, 195.1ms
Speed: 4.4ms preprocess, 195.1ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 9 cars, 1 bus, 187.1ms
Speed: 4.6ms preprocess, 187.1ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 9 cars, 2 buss, 271.6ms
Speed: 6.4ms preprocess, 271.6ms inference, 2.1ms postprocess per image at shape (1, 3

In [ ]:
import cv2
import json
import numpy as np
from ultralytics import YOLO

# ================= LOAD MODEL =================
model = YOLO("best.pt")

cap = cv2.VideoCapture("traffic.mp4")

# ================= ROI FUNCTION =================
def inside_roi(cx, cy, roi):
    x1, y1, x2, y2 = roi
    return x1 <= cx <= x2 and y1 <= cy <= y2

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    h, w, _ = frame.shape

    # ================= QUEUE ROI (60% of screen) =================
    QUEUE_ROI = (0, int(0.4 * h), w, h)

    roi_x1, roi_y1, roi_x2, roi_y2 = QUEUE_ROI
    ROI_AREA = (roi_x2 - roi_x1) * (roi_y2 - roi_y1)

    # Draw ROI
    cv2.rectangle(frame, (roi_x1, roi_y1), (roi_x2, roi_y2), (255, 255, 0), 2)
    cv2.putText(frame, "QUEUE ROI",
                (20, roi_y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                (255, 255, 0), 2)

    results = model.track(frame, persist=True, conf=0.4)

    queue_ids = set()
    occupied_area = 0

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy()

        for box, obj_id in zip(boxes, ids):
            x1, y1, x2, y2 = map(int, box)
            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)

            if inside_roi(cx, cy, QUEUE_ROI):
                queue_ids.add(int(obj_id))
                occupied_area += (x2 - x1) * (y2 - y1)
                color = (0, 0, 255)
            else:
                color = (0, 255, 0)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"ID {int(obj_id)}",
                        (x1, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                        color, 2)

    # ================= METRICS =================
    queue_length = len(queue_ids)
    queue_density = min(occupied_area / ROI_AREA, 1.0)

    # ================= DISPLAY =================
    cv2.putText(frame, f"Queue Length: {queue_length}",
                (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1,
                (0, 0, 255), 3)

    cv2.putText(frame, f"Queue Density: {queue_density:.2f}",
                (30, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 1,
                (255, 0, 0), 3)

    # ================= STREAMLIT DATA =================
    with open("queue_metrics.json", "w") as f:
        json.dump({
            "queue_length": queue_length,
            "queue_density": round(queue_density, 2)
        }, f)

    cv2.imshow("Traffic Queue System", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
